# Project K, Series Three: the surgery (activation patching), Qwen2.5-Omni-3B

**Why this notebook exists.** The tracing asked where a
sound gets lost inside the model. The helpers (tiny
classifiers trained on one layer's numbers at a time) only
read; the attention watch only reads. Round 2's readout,
with its controls at chance, said the sound's presence
stays readable at every layer and the answer barely looks
at the sound's positions. Reading cannot prove cause. The
surgery can, and it is the third instrument in the tracing
plan and the causal check on the readers.

**The surgery, in plain words.** Run a clip that carries
the sound (the donor) and save the numbers it produces at
one layer, at the sound's own positions in the row. Then
run a second clip (the recipient) with a hook set at that
same layer: the moment the layer finishes, the hook
overwrites the recipient's numbers at the sound's
positions with the donor's. Everything after that layer
runs on the patched numbers. If a silence run, patched
with a tone run's numbers, now answers yes to the beeping
question, then those numbers were carrying the tone, at
that layer, causally. Sweeping the layer from early to
late brackets where the sound stops being able to change
the answer.

**The design, fixed in advance.**

- **Clips.** 20 short WAVs, all exactly 4.000 seconds at
  16 kHz, sound alone, no picture: 8 tone, 8 silence,
  4 speech. They live in the Drive folder
  `tracing-qwen3b/patching_clips`, and Step 2 copies them
  here. The runner proves every clip against the folder's
  manifest before anything uses it: SHA-256 prefixes,
  seconds, and RMS, printed per clip, hard stop on any
  mismatch.
- **Questions.** The presence questions and the yes score
  are the Series Two runner's own, copied from its code:
  the tone and silence clips are asked "Is there the
  sound of beeping in this clip? Answer yes or no." and
  the speech clips "Is there the sound of a person
  speaking in this clip? Answer yes or no." The yes score
  is the probability of a yes token at the first generated
  position, from the same greedy generate call that
  produces the answer word. The open question, used only
  for a few context answers, is the swap study's P3.
- **Donor and recipient always run under the same
  question**, the direction's scored question, so a patch
  moves sound information, never question information.
- **Baselines first.** Every clip's unpatched yes score
  and answer word, so every patched score has its own
  recipient's starting point beside it.
- **The sweep.** Tone into silence (8 pairs, layers 0 to
  35) and speech into silence (4 pairs, all layers). Each
  patched run is one prefill and a first-token read; its
  row is appended to `patching_results.csv` the moment it
  finishes, so an interruption loses nothing.
- **Three controls.** Silence into tone (does the score
  fall?), tone into a different tone clip (a sham: the
  sound is already there, so little should change), and
  tone into silence patched at the non-audio positions
  only (the patch lands everywhere except the sound's own
  slots; no rescue is expected). A result that cannot
  survive its controls is not a result.
- **The open-form check.** At the single layer where the
  tone patch lifts the silence scores most (over the
  recipients' own baselines), the 8 patched silence runs
  also answer the open question, to test the registered
  prediction that naming is a later step than detection.

**Predictions, registered in the tracing plan before any
of this was built** (quoted exactly in the closing prints;
this notebook prints no verdict on them, the read happens
outside):

- Tone: patching rescues the yes/no answer at mid layers
  but not the open form.
- Speech: patching does NOT fix the denial, because
  nothing is missing: the loss sits in how the presence
  answer itself is formed, past the information.

**Where it runs.** On the free Colab account (kadjei1),
under the routing rule: the 3B is a small model. **How to
run it.** Run the cells from the top, in order. Step 1
sets up and proves the stack. Step 2 gets the patching
clips. Step 3 writes the runner and runs its check mode
(proof, loading candidates, layer resolution, the token
check, the feed proof). Step 4 runs battery mode:
baselines, captures, the sweep, the controls, and the
open-form check.

**The one rule.** We claim only what we ran: these clips,
this model, these layers, this run.

## Step 1 · Setup

The next cell installs the parts the model needs, with fixed version numbers
so the run is the same every time. It then prints the graphics chip it found.

**Before you run it:** the chip must be a T4. In the Colab menu, open
Runtime, then "Change runtime type", and pick the T4 GPU. On the free plan
this is the normal choice. If the cell prints "no GPU found", stop and change
the runtime type first. The model will not fit in plain memory.

The install takes a few minutes. That is normal.

One more setup step, new after run 2: the first code cell sets a memory
option (expandable segments) before torch loads. It lets the chip's memory
manager grow in pieces instead of reserving one big block, which helps when
the model already takes most of the chip. If you rerun cells in an old
session, restart the runtime once first, so the option takes effect from a
clean start.
**Patching addition.** At the end, the cell proves the
stack it just installed: it prints the imported
transformers version and the file it loaded from, and
stops the run if the version is not the pinned 4.52.4.
This build trains nothing and reads no attention
weights, so no further library proof is needed. This
notebook's model runs under the session's own Python,
so the versions printed here are the run's versions.


In [ ]:
# --- Setup -------------------------------------------------------------------
# First thing in this cell, before torch is loaded: ask the memory manager
# to use growing segments. This must be set BEFORE torch loads, which is
# why it sits above the torch import below. In a fresh runtime it takes
# effect. If you rerun this cell in a session where torch was already
# loaded, restart the runtime once and run from the top.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Fixed versions, so today's run matches tomorrow's run.
# soundfile and librosa are how the model's tools read the sound track
# out of each video file. Without them, the video would arrive silent,
# and this whole test would prove nothing.
import sys, subprocess

PACKAGES = [
    "transformers==4.52.4",
    "accelerate==1.8.1",
    "qwen-omni-utils==0.0.8",
    "soundfile==0.13.1",
    "librosa==0.11.0",
    "av",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PACKAGES,
               check=False)
print("install step done")

import torch
if torch.cuda.is_available():
    print("GPU found:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("no GPU found. Stop here. Set the runtime type to the T4 GPU,")
    print("then run this cell again.")
# --- Patching addition: prove the installed stack -------------------------
# The model runs under this same session Python, so the
# versions installed above are the versions the run uses.
# Prove them here, and stop if the transformers pin is
# not met.
import transformers
print("transformers version:", transformers.__version__)
print("transformers file:", transformers.__file__)
if transformers.__version__ != "4.52.4":
    raise SystemExit("STOP: transformers is "
                     + transformers.__version__
                     + ", not the pinned 4.52.4. Do not go on.")
print("stack proof passed: transformers 4.52.4, the Series One pin.")


## Step 2 · Get the patching clips

Twenty WAV files and their manifest, from your Google
Drive folder `tracing-qwen3b/patching_clips`:

- `tone_00.wav` to `tone_07.wav`: the tone clips.
- `silence_00.wav` to `silence_07.wav`: digital silence.
- `speech_00.wav` to `speech_03.wav`: the probe sentence.
- `manifest.json`: each file's SHA-256 prefix and its
  length in seconds, the record the runner's feed proof
  checks the session's copies against.

The next cell links your Drive, finds that folder, and
copies all 21 files to a plain work folder here. The
runner re-measures everything from inside the session
before it believes any of it.

Do not go on until every file says "ready". If a file is
missing, the cell stops and names it.

In [ ]:
# --- Get the patching clips -------------------------------------------------------
import os, shutil, subprocess

FILES = (["manifest.json"]
         + [f"tone_{i:02d}.wav" for i in range(8)]
         + [f"silence_{i:02d}.wav" for i in range(8)]
         + [f"speech_{i:02d}.wav" for i in range(4)])
WORK = "/content/patching_clips"
DRIVE_CLIPS = "/content/drive/MyDrive/tracing-qwen3b/patching_clips"
os.makedirs(WORK, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)

SOURCES = [DRIVE_CLIPS, "/content/tracing-qwen3b/patching_clips"]

def bring(name):
    dest = os.path.join(WORK, name)
    if os.path.exists(dest):
        return True
    for folder in SOURCES:
        src = os.path.join(folder, name)
        if os.path.exists(src):
            shutil.copy(src, dest)
            print("copied", name, "from", folder)
            return True
    return False

def duration_s(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", path], capture_output=True, text=True)
    try:
        return float(out.stdout.strip())
    except Exception:
        return None

missing = []
for name in FILES:
    bring(name)
    path = os.path.join(WORK, name)
    if os.path.exists(path):
        d = duration_s(path)
        extra = f", {d:.3f} s" if d is not None else ""
        print("ready:  ", name, f"({os.path.getsize(path)} bytes{extra})")
    else:
        missing.append(name)
        print("MISSING:", name)
if missing:
    print()
    print("!" * 72)
    print("STOP. These patching files are missing:")
    for name in missing:
        print("   ", name)
    print("Check the Drive folder is tracing-qwen3b/patching_clips, or")
    print("upload the files by hand, then run this cell again.")
    print("Do not go on to Step 3.")
    print("!" * 72)
    raise SystemExit("STOP: patching files missing in Step 2")
print()
print("ready: all 21 patching files (20 clips and the manifest)")

## Step 3 · Write the runner, then check: proof, candidates, layers, token check, feed proof

The model never runs inside this notebook's cells directly.
It runs in a plain script, `/content/qwen_patching_run.py`,
under this session's own Python, the Qwen notebooks'
pattern. The first cell below writes that script. The
second runs it in **check mode**.

Check mode does five things, in order:

1. **Proof again.** The versions and files the runner
   imported, with the transformers pin checked a second
   time: exactly 4.52.4, or the run stops. One note
   explains this build's single deliberate difference from
   the tracing runners' load call: no eager attention,
   because nothing here reads attention weights, so the
   model loads with the library default, the door the
   Series Two asks ran through.
2. **The loading candidates.** The tracing runners' list:
   the 3B in 16-bit form (float16) first, the 7B AWQ build
   second, only if the small one fails. The talker is
   turned off on whichever loads. A failed candidate
   prints its full traceback and appends the same words to
   `patching_load_log.txt` in the tracing folder.
3. **Layer resolution.** Every module list in the loaded
   model is printed, and the decoder is the list of
   exactly 36 modules under the thinker. The resolved path
   and count print; anything else stops the run before a
   single hook is set. This is resolved from the loaded
   model, never assumed from a version's layout.
4. **The token check.** One tone clip and one speech clip
   from the patching set go through the processor. For
   each, the runner prints the full histogram of token ids
   in the processed row (every id, its count, its decoded
   name, special tokens flagged) and the processor's audio
   feature lengths. The audio token id is derived from the
   histogram: the candidate that occurs at audio-frame
   scale. Both clips must derive the same id, and its
   count must be exactly 100, the count these 4.0 second
   clips are built to produce, or the step prints TOKEN
   CHECK FAILED and stops the run (exit code 11) before
   any baseline. One forward pass then cross-checks the
   hidden states' layer count against the resolved list.
   The whole transcript is saved as
   `patching_token_check.txt`.
5. **The feed proof.** Every one of the 20 clips is
   checked against `manifest.json`: the file's SHA-256
   prefix, its decoded length in seconds, its sample rate
   and RMS, printed per clip. Any mismatch stops the run
   (exit code 7). Nothing is ever scored on an unproven
   clip.

No baselines are taken and no patch is run in check mode.

In [ ]:
# --- Write the patching runner script --------------------------------------------
# The runner holds the proof, the loading candidates, the
# layer resolution, the token check, the feed proof, the
# baselines, the captures, the patching sweep, and the
# open-form check. It is written to a plain file here and
# only ever run under this session's own Python, the
# interpreter Step 1 installed into.
RUNNER_SOURCE = '\n# qwen_patching_run.py - the surgery (activation patching)\n# runner for Qwen2.5-Omni-3B, Series Three test 2. Written\n# by project_k_qwen_patching_colab.ipynb. It runs under the\n# notebook session\'s own Python: the Qwen notebooks install\n# their stack into the session (Step 1), and there is no\n# private environment to run under. Modes: "check" runs the\n# proof, the loading candidates, the layer resolution, the\n# TOKEN CHECK, and the FEED PROOF, then stops. "battery"\n# does all of that and then runs the baselines, the donor\n# captures, the patching sweep with its controls, and the\n# best-layer open-form check.\n#\n# Exit codes, one per failure family, all loud:\n#   3  the stack proof failed (transformers is not 4.52.4)\n#   4  both loading candidates failed\n#   7  the feed proof failed (a clip missing, its SHA-256\n#      prefix off the manifest, its length off, or the feed\n#      path returned no audio or zeros on a sound clip)\n#   11 the token check failed: the audio positions could\n#      not be settled from the histogram, or their count is\n#      not the expected 100\n#   12 a patching mechanic failed: the decoder layer list\n#      did not resolve to exactly 36 layers, the hidden\n#      states disagree with it, or a donor and a recipient\n#      row do not line up for a patch\nimport os\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nimport sys\n\nMODE = sys.argv[1] if len(sys.argv) > 1 else "check"\nprint("runner mode:", MODE)\nprint("runner python:", sys.version.split()[0], sys.executable)\n\n# --- The fixed seed ------------------------------------------------------------------\n# Generation is greedy, so the seed decides nothing about\n# the answers; it is fixed and printed anyway, the house\n# rule, so every random draw anywhere in the run (there\n# should be none that matter) is reproducible.\nimport random\nimport numpy as np\nimport torch\n\nSEED = 20261105\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntorch.manual_seed(SEED)\nprint("fixed seed:", SEED)\n\n# --- Proof lines, printed before anything loads --------------------------------\nimport transformers\nprint("torch version:", torch.__version__)\nprint("transformers version:", transformers.__version__)\nprint("transformers file:", transformers.__file__)\nimport qwen_omni_utils\nprint("qwen_omni_utils file:", qwen_omni_utils.__file__)\nif transformers.__version__ != "4.52.4":\n    print("STOP: this script is running on transformers",\n          transformers.__version__, "and needs exactly 4.52.4,")\n    print("the version the Series One Qwen notebooks pin.")\n    sys.exit(3)\nprint("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()\n      else "none")\nprint("attention implementation note: the loading candidates below pass")\nprint("no attn_implementation, so the model loads with the library default")\nprint("(sdpa), the same door the Series Two asks ran through. This build")\nprint("reads no attention weights anywhere, so it does not need the eager")\nprint("mode tracing round 2 had to load for its attention readout.")\n\n# --- Homes for the outputs -------------------------------------------------------\n# The patching outputs live in the Drive folder\n# tracing-qwen3b beside the tracing rounds\' outputs, with\n# /content as the second home, the house pattern.\nDRIVE_TRACING = "/content/drive/MyDrive/tracing-qwen3b"\nLOCAL_TRACING = "/content/tracing-qwen3b"\nfor d in (DRIVE_TRACING, LOCAL_TRACING):\n    try:\n        os.makedirs(d, exist_ok=True)\n    except Exception:\n        pass\n\ndef out_paths(name):\n    return [os.path.join(DRIVE_TRACING, name),\n            os.path.join(LOCAL_TRACING, name)]\n\ndef save_both(name, write_fn):\n    saved = []\n    for p in out_paths(name):\n        try:\n            write_fn(p)\n            saved.append(p)\n            print("saved:", p)\n        except Exception as e:\n            print("could not save", p, "-", str(e)[:160])\n    return saved\n\n# --- The load log ------------------------------------------------------------------\nimport traceback, datetime\n\nLOAD_LOG_PATHS = out_paths("patching_load_log.txt")\n\ndef log_load(text):\n    for p in LOAD_LOG_PATHS:\n        try:\n            with open(p, "a") as fh:\n                fh.write(text)\n        except Exception:\n            pass   # that home is not mounted; the printed copy stands\n\ndef log_paths_saved():\n    return [p for p in LOAD_LOG_PATHS if os.path.exists(p)]\n\nlog_load("=" * 70 + "\\n")\nlog_load("Qwen2.5-Omni-3B patching (Series Three surgery) load log - "\n         + datetime.datetime.now().isoformat(timespec="seconds")\n         + " - runner mode: " + MODE + "\\n")\n\n# --- Load the model: the tracing runners\' candidates ------------------------------\n# Same candidates, same order, same talker-off call as the\n# tracing runners. The one printed difference is the\n# missing attn_implementation (see the proof lines above\n# for why).\nfrom transformers import (Qwen2_5OmniForConditionalGeneration,\n                          Qwen2_5OmniProcessor)\n\ndef free_memory():\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nmodel = None\nprocessor = None\nMODEL_NAME = None   # repo name of the model that loaded\nLOAD_MODE = None    # the fit that worked\n\nCANDIDATES = [("Qwen/Qwen2.5-Omni-3B", "fp16"),\n              ("Qwen/Qwen2.5-Omni-7B-AWQ", "AWQ weights, fp16 load")]\n\ncandidate_errors = []\nfor repo, mode in CANDIDATES:\n    print("=" * 70)\n    print(f"candidate: {repo} ({mode})")\n    try:\n        m = Qwen2_5OmniForConditionalGeneration.from_pretrained(\n            repo, torch_dtype=torch.float16, device_map="auto")\n        m.disable_talker()  # written answers only; saves memory\n        p = Qwen2_5OmniProcessor.from_pretrained(repo)\n        model, processor = m, p\n        MODEL_NAME, LOAD_MODE = repo, mode\n        print("loaded:", repo)\n        break\n    except Exception:\n        tb = traceback.format_exc()\n        candidate_errors.append((repo, mode, tb))\n        print("this candidate failed. Full traceback:")\n        print(tb)\n        log_load("-" * 70 + "\\n")\n        log_load("candidate failed: " + repo + " (" + mode + ")\\n")\n        log_load(tb)\n        model = None\n        processor = None\n        free_memory()\n\nprint()\nif model is None:\n    print("!" * 70)\n    print("BOTH CANDIDATES FAILED. No model loaded.")\n    saved = log_paths_saved()\n    if saved:\n        print("The full traceback of every candidate is saved, word")\n        print("for word, in:")\n        for p in saved:\n            print("   ", p)\n    else:\n        print("The load log could not be written anywhere; the full")\n        print("tracebacks are in the output above, and nowhere else.")\n    print("Do not go on. Show this whole output to Muse.")\n    print("!" * 70)\n    sys.exit(4)\nprint("MODEL FOR THIS RUN:", MODEL_NAME, "in", LOAD_MODE, "form")\nlog_load("loaded: " + MODEL_NAME + " in " + LOAD_MODE + " form\\n")\nprint("environment: Colab session python (Step 1 installs),",\n      "transformers 4.52.4")\nif torch.cuda.is_available():\n    used = torch.cuda.memory_allocated() / 1e9\n    print(f"chip memory already used by the model: {used:.1f} GB")\n\nTHINKER = model.thinker\nprint("thinker:", type(THINKER).__name__)\n\n# --- Resolve the thinker decoder layers --------------------------------------------\n# The patch hooks live on the thinker\'s decoder layers, so\n# the layer list is resolved from the loaded model at\n# runtime, never assumed from a version\'s source layout:\n# every module list in the model is printed, the list of\n# exactly 36 modules under the thinker is the decoder, and\n# anything else stops the run (exit 12) before a single\n# hook is set. Sweep layer L means the output of decoder\n# layer L, which is hidden_states[L + 1] in a forward\'s\n# returned tuple (index 0 is the embedding output).\nN_LAYERS_EXPECTED = 36\nLAYER_PATH = None\nLAYERS = None\n\ndef mechanics_fail(reason):\n    print()\n    print("!" * 70)\n    print("PATCHING MECHANICS FAILED: " + reason)\n    print("No hook has been trusted and no patch score recorded.")\n    print("Show this whole output to Muse.")\n    print("!" * 70)\n    sys.exit(12)\n\nall_lists = [(name, mod) for name, mod in model.named_modules()\n             if isinstance(mod, torch.nn.ModuleList)]\nprint("module lists found in the loaded model:")\nfor name, mod in all_lists:\n    print(f"  {name}: {len(mod)} modules")\nexact = [(name, mod) for name, mod in all_lists\n         if len(mod) == N_LAYERS_EXPECTED]\nthinker_exact = [(name, mod) for name, mod in exact\n                 if name.startswith("thinker")]\nchosen = thinker_exact if thinker_exact else exact\nif len(chosen) != 1:\n    mechanics_fail(\n        f"the decoder layer list did not resolve to exactly one "\n        f"list of {N_LAYERS_EXPECTED} modules (found "\n        f"{len(exact)} such lists, {len(thinker_exact)} of them "\n        f"under the thinker); the printed module lists above "\n        f"are the evidence")\nLAYER_PATH, _layers_mod = chosen[0]\nLAYERS = list(_layers_mod)\nprint(f"decoder layer list resolved: {LAYER_PATH}, "\n      f"{len(LAYERS)} layers")\nprint(f"layer module type: {type(LAYERS[0]).__name__}")\nlog_load("decoder layer list: " + LAYER_PATH + ", "\n         + str(len(LAYERS)) + " layers\\n")\n\n# --- The yes score: the Series Two runner\'s own scorer ------------------------------\n# Copied from build_series2_qwen_notebook.py, the Qwen\n# Series Two runner: the yes token ids are read from the\n# model\'s own tokenizer (the first token of "Yes" and of\n# " Yes", lines 830-837 there), and yes_probability (lines\n# 995-1010 there) takes the softmax over the vocabulary of\n# the first generated position\'s logits, from the first\n# score row of the same greedy generate call that produced\n# the answer, summed over those ids. The generate call\n# itself (in ask_generate below) is that runner\'s ask path\n# (lines 1171-1178 there): thinker_max_new_tokens,\n# do_sample=False, return_dict_in_generate=True,\n# output_scores=True, and only the newly written tokens\n# are decoded.\nYES_TOKEN_IDS = []\nfor _word in ("Yes", " Yes"):\n    _enc = processor.tokenizer.encode(_word, add_special_tokens=False)\n    if _enc:\n        YES_TOKEN_IDS.append(int(_enc[0]))\nYES_TOKEN_IDS = sorted(set(YES_TOKEN_IDS))\nprint("yes token ids for scoring:", YES_TOKEN_IDS,\n      [processor.tokenizer.decode([i]) for i in YES_TOKEN_IDS])\n\ndef yes_probability(scores):\n    if not scores:\n        return None\n    logits = scores[0][0].float()\n    probs = torch.softmax(logits, dim=-1)\n    total = 0.0\n    for i in YES_TOKEN_IDS:\n        total += float(probs[i].item())\n    return total\n\n# --- The feed path -----------------------------------------------------------------\n# Sound alone, the tracing runners\' feed form: the\n# conversation carries an audio element naming the WAV file\n# and a text element, and no image and no video element.\n# process_mm_info decodes the audio element with the same\n# librosa call at 16 kHz mono it uses for a video\n# soundtrack.\nimport json, math\nimport librosa\nimport soundfile as sf\nfrom qwen_omni_utils import process_mm_info\n\nPATCH_CLIPS = "/content/patching_clips"\nPROMPT_BEEPS = \'Is there the sound of beeping in this clip? Answer yes or no.\'\nPROMPT_SPEECH = \'Is there the sound of a person speaking in this clip? Answer yes or no.\'\nPROMPT_OPEN = "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'."\nQUESTION_FOR = {"tone": PROMPT_BEEPS, "silence": PROMPT_BEEPS,\n                "speech": PROMPT_SPEECH}\nTARGET_FOR = {"tone": \'beeping\', "silence": \'beeping\',\n              "speech": \'a person speaking\'}\n\nclass FeedError(RuntimeError):\n    pass\n\ndef conversation_for(path, prompt):\n    return [{\n        "role": "user",\n        "content": [\n            {"type": "audio", "audio": path},\n            {"type": "text", "text": prompt},\n        ],\n    }]\n\ndef feed_inputs(path, prompt, expect_silence=False):\n    # Run the feed path for one clip and build the model\'s\n    # inputs. Raises FeedError if the path returns no audio:\n    # a missing track stops the run, it is never padded. A\n    # track that decodes to all zeros also stops the run,\n    # unless the caller declares the clip silent by design\n    # (a patching silence clip): an all-zeros feed on a clip\n    # that should carry sound is a broken feed, not a quiet\n    # clip.\n    if not os.path.exists(path):\n        raise FeedError(f"feed file missing: {path}")\n    conversation = conversation_for(path, prompt)\n    audios, images, videos = process_mm_info(\n        conversation, use_audio_in_video=False)\n    if audios is None or len(audios) == 0 or audios[0] is None:\n        raise FeedError(\n            f"the feed path returned no audio for {path}")\n    audio_np = np.asarray(audios[0], dtype=np.float32)\n    if len(audio_np) == 0:\n        raise FeedError(\n            f"decoded track is empty for {path}")\n    if not expect_silence and not np.any(audio_np != 0):\n        raise FeedError(\n            f"decoded track is all zeros for {path}, on a clip "\n            f"that should carry sound")\n    text = processor.apply_chat_template(\n        conversation, add_generation_prompt=True, tokenize=False)\n    inputs = processor(text=text, audio=audios,\n                       return_tensors="pt", padding=True)\n    inputs = inputs.to(model.device).to(model.dtype)\n    return inputs, audio_np\n\ndef thinker_forward(inputs):\n    # One frozen pass through the thinker with every layer\'s\n    # numbers saved. The model is never trained; this is a\n    # reading, not a conversation.\n    with torch.no_grad():\n        out = THINKER(**inputs, output_hidden_states=True,\n                      output_attentions=False)\n    return out\n\n# --- The token check -----------------------------------------------------------------\n# The tracing round 2 design, with the assert the brief for\n# this build fixes: for one tone clip and one speech clip\n# from the patching set, print the FULL histogram of token\n# ids in the processed input row (every id with count > 0,\n# its token name and decoded form, special tokens flagged),\n# print the processor\'s audio feature lengths, count every\n# candidate audio id in the row, and DERIVE the audio token\n# id from the histogram: the candidate with the highest\n# count, the one that occurs at audio-frame scale. Both\n# clips must derive the same id, and its count must be\n# exactly EXPECTED_AUDIO_POSITIONS (100 for these 4.0 s\n# clips), or the step prints TOKEN CHECK FAILED and stops\n# the run (exit 11) before any baseline or patch. The audio\n# only rows here are shorter than the tracing rounds\' rows\n# (no picture tokens); nothing about the totals is assumed.\n# One thinker forward on the tone clip then confirms the\n# hidden states come back and cross-checks the layer count\n# against the resolved decoder list. Every line is saved\n# to patching_token_check.txt, pass or fail.\nfrom collections import Counter\n\nEXPECTED_AUDIO_POSITIONS = 100\nAUDIO_TOKEN_ID = None   # set by the token check, from the histogram\nN_STATES = None\nHIDDEN_SIZE = None\nTOKEN_CHECK_LINES = []\n\ndef tc(line=""):\n    print(line)\n    TOKEN_CHECK_LINES.append(line)\n\ndef save_token_check():\n    text = "\\n".join(TOKEN_CHECK_LINES) + "\\n"\n    def w(p):\n        with open(p, "w") as fh:\n            fh.write(text)\n    save_both("patching_token_check.txt", w)\n\ndef token_check_fail(reason):\n    tc()\n    tc("!" * 70)\n    tc("TOKEN CHECK FAILED: " + reason)\n    tc("The audio positions are not settled, so no baseline")\n    tc("has run and no patch score in this run can be trusted.")\n    tc("The full histogram above is the evidence. Show this whole")\n    tc("output to Muse.")\n    tc("!" * 70)\n    save_token_check()\n    sys.exit(11)\n\ndef audio_candidates():\n    # The candidate audio ids, from the model\'s own\n    # tokenizer and config, never a guessed number. Same\n    # evidence the tracing checks gathered; the winner is\n    # picked from the histogram (see token_check).\n    tok = processor.tokenizer\n    vocab = tok.get_vocab()\n    audio_named = {t: i for t, i in vocab.items() if "audio" in t}\n    tc("tokenizer tokens whose text mentions audio:")\n    for t in sorted(audio_named):\n        tc(f"  {t!r} -> id {audio_named[t]}")\n    candidates = {}\n    pad_id = tok.convert_tokens_to_ids("<|audio_pad|>")\n    if pad_id is not None and int(pad_id) >= 0:\n        candidates[int(pad_id)] = "<|audio_pad|> (converted by name)"\n    for t, i in audio_named.items():\n        candidates[int(i)] = f"tokenizer token {t!r}"\n    for holder_name, holder in (\n            ("model.config", model.config),\n            ("thinker_config",\n             getattr(model.config, "thinker_config", None))):\n        if holder is None:\n            continue\n        for attr in ("audio_token_id", "audio_token_index"):\n            val = getattr(holder, attr, None)\n            if val is not None:\n                tc(f"  {holder_name}.{attr} = {val}")\n                candidates.setdefault(\n                    int(val), f"config {holder_name}.{attr}")\n    return candidates\n\ndef histogram_for(label, path, prompt, candidates):\n    # Feed one patching clip and print its full token\n    # histogram plus the processor\'s audio feature lengths.\n    # Returns (inputs, chosen_id, chosen_count, row_length).\n    tok = processor.tokenizer\n    try:\n        inputs, audio_np = feed_inputs(path, prompt)\n    except FeedError as e:\n        token_check_fail(\n            f"the {label} clip\'s feed failed inside the token "\n            f"check: {e}")\n    row = [int(x) for x in inputs.input_ids[0].tolist()]\n    tc("-" * 70)\n    tc(f"TOKEN HISTOGRAM: {label} clip")\n    tc(f"  file: {path}")\n    tc(f"  decoded waveform: {len(audio_np)} samples at 16 kHz "\n       f"({len(audio_np) / 16000:.2f} s)")\n    tc(f"  sequence length: {len(row)} positions")\n    counts = Counter(row)\n    tc(f"  distinct token ids: {len(counts)}")\n    tc("  full histogram, most common first (every id with "\n       "count > 0):")\n    special_ids = set(int(x) for x in tok.all_special_ids)\n    for tid, cnt in counts.most_common():\n        name = tok.convert_ids_to_tokens([tid])[0]\n        decoded = tok.decode([tid])\n        flag = " [special token]" if tid in special_ids else ""\n        tc(f"    id {tid}: count {cnt}, token name {name!r}, "\n           f"decodes to {decoded!r}{flag}")\n    feats = getattr(inputs, "input_features", None)\n    if feats is not None:\n        tc(f"  processor input_features shape: "\n           f"{tuple(feats.shape)} (batch, mel bins, audio frames)")\n    else:\n        tc("  processor input_features: not present on the inputs")\n    fam = getattr(inputs, "feature_attention_mask", None)\n    if fam is not None:\n        tc(f"  processor feature_attention_mask shape: "\n           f"{tuple(fam.shape)}, true frames per row: "\n           f"{[int(x) for x in fam.sum(dim=-1).tolist()]}")\n    else:\n        tc("  processor feature_attention_mask: not present")\n    tc("  candidate audio ids, counted in this row:")\n    cand_counts = {}\n    for cid, why in sorted(candidates.items()):\n        cnt = row.count(cid)\n        cand_counts[cid] = cnt\n        tc(f"    candidate id {cid} ({why}): count {cnt}, "\n           f"decodes to {tok.decode([cid])!r}")\n    chosen = max(cand_counts, key=lambda c: cand_counts[c])\n    positions = [i for i, x in enumerate(row) if x == chosen]\n    tc(f"  DERIVED audio token id for the {label} clip: {chosen} "\n       f"({candidates[chosen]}), decodes to "\n       f"{tok.decode([chosen])!r}")\n    tc(f"  audio positions: {len(positions)} of {len(row)}; "\n       f"first {positions[:5]}, last {positions[-5:]}")\n    return inputs, chosen, cand_counts[chosen], len(row)\n\ndef token_check():\n    global AUDIO_TOKEN_ID, N_STATES, HIDDEN_SIZE\n    tc("=" * 70)\n    tc("TOKEN CHECK: settle the audio positions from the "\n       "histogram, before any baseline or patch")\n    tc(f"expected audio-position count for these 4.0 s clips: "\n       f"{EXPECTED_AUDIO_POSITIONS}")\n    candidates = audio_candidates()\n    tone_path = os.path.join(PATCH_CLIPS, "tone_00.wav")\n    speech_path = os.path.join(PATCH_CLIPS, "speech_00.wav")\n    tone_inputs, tone_id, tone_count, tone_len = histogram_for(\n        "tone", tone_path, PROMPT_BEEPS, candidates)\n    _, speech_id, speech_count, speech_len = histogram_for(\n        "speech", speech_path, PROMPT_SPEECH, candidates)\n    if tone_id != speech_id:\n        token_check_fail(\n            f"the two clips derive different audio token ids "\n            f"(tone {tone_id}, speech {speech_id}); the audio "\n            f"positions cannot be settled on a split answer")\n    if tone_count != EXPECTED_AUDIO_POSITIONS \\\n            or speech_count != EXPECTED_AUDIO_POSITIONS:\n        token_check_fail(\n            f"the derived audio-position count is not the "\n            f"expected {EXPECTED_AUDIO_POSITIONS} (tone "\n            f"{tone_count}, speech {speech_count}); patching "\n            f"replaces every audio position, so a count that "\n            f"differs from the design stops the run")\n    AUDIO_TOKEN_ID = tone_id\n    tc("-" * 70)\n    tc(f"AUDIO TOKEN ID SETTLED: {AUDIO_TOKEN_ID}, decodes to "\n       f"{processor.tokenizer.decode([AUDIO_TOKEN_ID])!r}")\n    tc(f"audio positions: tone {tone_count} of {tone_len} row "\n       f"positions; speech {speech_count} of {speech_len}")\n    out = thinker_forward(tone_inputs)\n    hs = getattr(out, "hidden_states", None)\n    if hs is None or len(hs) == 0:\n        token_check_fail(\n            "the thinker returned no hidden states on the tone "\n            "clip; there would be nothing to capture or patch")\n    N_STATES = len(hs)\n    HIDDEN_SIZE = int(hs[0].shape[2])\n    tc(f"hidden states returned: {N_STATES} tensors (index 0 is "\n       f"the embedding output, then one per decoder layer)")\n    tc(f"hidden size: {HIDDEN_SIZE}")\n    if N_STATES - 1 != len(LAYERS):\n        tc(f"hidden states imply {N_STATES - 1} decoder layers "\n           f"but the resolved layer list holds {len(LAYERS)}")\n        save_token_check()\n        mechanics_fail(\n            "the hidden states and the resolved decoder layer "\n            "list disagree on the layer count")\n    tc("TOKEN CHECK PASSED: the audio positions are derived "\n       "from the histogram, counted, and the same for both "\n       "clips. Every capture and patch in this run works on "\n       "these positions.")\n    save_token_check()\n    del out, tone_inputs\n    free_memory()\n\ntoken_check()\n\ndef audio_positions(input_ids_row):\n    row = [int(x) for x in input_ids_row]\n    return [i for i, x in enumerate(row) if x == AUDIO_TOKEN_ID]\n\n# --- The feed proof ------------------------------------------------------------------\n# Every clip the run uses is proven against the clips\'\n# manifest before any baseline or patch: the file\'s own\n# SHA-256 prefix must match the manifest\'s sha256_16, the\n# decoded length must match the manifest\'s seconds, and the\n# sample rate and RMS are printed per clip. The manifest\'s\n# hashes are of the file bytes (verified against the files\n# when this notebook was built). Any mismatch prints FEED\n# PROOF FAILED and stops the run (exit 7): nothing is ever\n# scored on an unproven clip.\nimport csv, hashlib\n\ndef rms_db_np(x):\n    if len(x) == 0:\n        return -120.0\n    v = float(np.sqrt(np.mean(np.square(x.astype(np.float64)))))\n    return 20.0 * math.log10(v) if v > 0 else -120.0\n\ndef file_sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as fh:\n        for block in iter(lambda: fh.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\nCLIP_INFO = {}   # clip name (no .wav) -> record\n\ndef feed_proof_fail(reason):\n    print()\n    print("!" * 70)\n    print("FEED PROOF FAILED: " + reason)\n    print("No baseline has been taken and no patch scored on an")\n    print("unproven clip. Show this whole output to Muse.")\n    print("!" * 70)\n    sys.exit(7)\n\ndef feed_proof():\n    manifest_path = os.path.join(PATCH_CLIPS, "manifest.json")\n    if not os.path.exists(manifest_path):\n        feed_proof_fail(f"the clips\' manifest is missing: "\n                        f"{manifest_path}")\n    with open(manifest_path) as fh:\n        manifest = json.load(fh)\n    print("=" * 70)\n    print(f"FEED PROOF: {len(manifest)} clips against the manifest")\n    for entry in manifest:\n        name = entry["file"]\n        clip = name[:-4] if name.endswith(".wav") else name\n        group = clip.split("_")[0]\n        path = os.path.join(PATCH_CLIPS, name)\n        if not os.path.exists(path):\n            feed_proof_fail(f"clip file missing: {path}")\n        digest = file_sha256(path)\n        if not digest.startswith(entry["sha256_16"]):\n            feed_proof_fail(\n                f"{name}: the file\'s SHA-256 begins "\n                f"{digest[:16]}, the manifest says "\n                f"{entry[\'sha256_16\']}; this is not the clip "\n                f"the manifest describes")\n        wave, sr = sf.read(path, dtype="float32")\n        if wave.ndim > 1:\n            wave = wave[:, 0]\n        seconds = len(wave) / sr if sr else 0.0\n        if abs(seconds - float(entry["seconds"])) > 0.001:\n            feed_proof_fail(\n                f"{name}: decoded length {seconds:.3f} s, the "\n                f"manifest says {entry[\'seconds\']} s")\n        rms = rms_db_np(wave)\n        CLIP_INFO[clip] = {"clip": clip, "group": group,\n                           "file": name, "path": path,\n                           "seconds": seconds, "sr": int(sr),\n                           "rms_db": round(rms, 2)}\n        print(f"  feed proof {name}: {seconds:.3f} s, "\n              f"{len(wave)} samples at {sr} Hz, RMS {rms:.2f} dB, "\n              f"sha256 prefix {digest[:16]} matches the manifest")\n    print(f"feed proof passed: {len(CLIP_INFO)} clips proven "\n          f"against the manifest")\n\nfeed_proof()\n\nif MODE == "check":\n    print()\n    print("check mode done: proof, candidates, layer resolution,")\n    print("the token check, and the feed proof all passed. No")\n    print("baselines were taken and no patch was run.")\n    print("Run Step 4 for the battery.")\n    sys.exit(0)\n\n# --- Ask machinery ---------------------------------------------------------------------\n# One generate call shape for every answer in this build,\n# the Series Two ask path\'s audio alone arm: the processor\n# builds the inputs from the conversation, and the wrapper\n# generate runs greedy with the cap passed as\n# thinker_max_new_tokens, returning its scores so a yes/no\n# ask records the yes probability from the first score row\n# of the same call that produced the answer. Only the newly\n# written tokens are decoded.\nPRESENCE_MAX_TOKENS = 16\nOPEN_MAX_TOKENS = 300\nRESULTS_FIELDS = [\'pair\', \'donor\', \'recipient\', \'direction\', \'layer\', \'yes_score\', \'answer_word\']\nBASELINE_FIELDS = [\'clip\', \'question_target\', \'prompt\', \'form\', \'yes_score\', \'answer\']\nOPENFORM_FIELDS = [\'pair\', \'layer\', \'answer\']\n\ndef build_inputs(clip, prompt):\n    info = CLIP_INFO[clip]\n    try:\n        inputs, _ = feed_inputs(\n            info["path"], prompt,\n            expect_silence=(info["group"] == "silence"))\n    except FeedError as e:\n        print()\n        print("!" * 70)\n        print("FEED HARD STOP:")\n        print(str(e))\n        print(f"Clip {clip} was not run, and the run stops here.")\n        print("No padding was substituted. Show this whole output")\n        print("to Muse.")\n        print("!" * 70)\n        sys.exit(7)\n    return inputs\n\ndef generate_from_inputs(inputs, max_tokens, want_yes):\n    with torch.no_grad():\n        out = model.generate(\n            **inputs, thinker_max_new_tokens=max_tokens,\n            do_sample=False, return_dict_in_generate=True,\n            output_scores=True)\n    new_tokens = out.sequences[:, inputs.input_ids.shape[1]:]\n    decoded = processor.batch_decode(\n        new_tokens, skip_special_tokens=True,\n        clean_up_tokenization_spaces=False)[0].strip()\n    yp = yes_probability(out.scores) if want_yes else None\n    del out, new_tokens\n    return decoded, yp\n\ndef ask_unpatched(clip, prompt, max_tokens, want_yes):\n    inputs = build_inputs(clip, prompt)\n    decoded, yp = generate_from_inputs(inputs, max_tokens, want_yes)\n    del inputs\n    free_memory()\n    return decoded, yp\n\n# --- Baselines ---------------------------------------------------------------------------\n# Every clip\'s unpatched presence score and answer word\n# under its own question (tone and silence under the\n# beeping question, speech under the speaking question),\n# then the open question on three clips as context. Saved\n# whole to patching_baselines.csv; every later patched\n# score is read against its own recipient\'s baseline.\nprint("=" * 70)\nprint("BASELINES: unpatched, every clip")\nbaseline_rows = []\nbaseline_yes = {}\nfor clip in ([\'tone_00\', \'tone_01\', \'tone_02\', \'tone_03\', \'tone_04\', \'tone_05\', \'tone_06\', \'tone_07\'] + [\'silence_00\', \'silence_01\', \'silence_02\', \'silence_03\', \'silence_04\', \'silence_05\', \'silence_06\', \'silence_07\'] + [\'speech_00\', \'speech_01\', \'speech_02\', \'speech_03\']):\n    prompt = QUESTION_FOR[CLIP_INFO[clip]["group"]]\n    decoded, yp = ask_unpatched(clip, prompt, PRESENCE_MAX_TOKENS,\n                                True)\n    baseline_yes[clip] = yp\n    baseline_rows.append({\n        "clip": clip, "question_target": TARGET_FOR[\n            CLIP_INFO[clip]["group"]],\n        "prompt": prompt, "form": "presence",\n        "yes_score": f"{yp:.4f}", "answer": decoded})\n    print(f"  baseline {clip}: yes {yp:.4f}, answer {decoded!r}")\nfor clip in ("tone_00", "silence_00", "speech_00"):\n    decoded, _ = ask_unpatched(clip, PROMPT_OPEN, OPEN_MAX_TOKENS,\n                                False)\n    baseline_rows.append({\n        "clip": clip, "question_target": "", "prompt": PROMPT_OPEN,\n        "form": "open_p3", "yes_score": "", "answer": decoded})\n    print(f"  baseline open {clip}: {decoded!r}")\n\ndef write_baselines(p):\n    with open(p, "w", newline="") as fh:\n        w = csv.DictWriter(fh, fieldnames=BASELINE_FIELDS)\n        w.writeheader()\n        w.writerows(baseline_rows)\nsave_both("patching_baselines.csv", write_baselines)\n\n# --- Donor captures ------------------------------------------------------------------------\n# A capture is one frozen thinker forward on a donor clip\n# under the direction\'s question: the full hidden states\n# (every layer\'s output, index 0 the embedding output),\n# kept on the CPU as float16, with the input row and its\n# audio positions. Patching at sweep layer L uses index\n# L + 1, the output of decoder layer L. Captures are taken\n# once per donor per question and reused for every layer\n# and every recipient in that direction.\nprint("=" * 70)\nprint("CAPTURES: donor runs, one per donor per question")\n\ndef capture(clip, prompt):\n    inputs = build_inputs(clip, prompt)\n    row = [int(x) for x in inputs.input_ids[0].tolist()]\n    pos = audio_positions(row)\n    if len(pos) != EXPECTED_AUDIO_POSITIONS:\n        mechanics_fail(\n            f"donor {clip} has {len(pos)} audio positions, not "\n            f"the {EXPECTED_AUDIO_POSITIONS} the token check "\n            f"settled; a capture on unsettled positions cannot "\n            f"be patched from")\n    out = thinker_forward(inputs)\n    hs = getattr(out, "hidden_states", None)\n    if hs is None or len(hs) != len(LAYERS) + 1:\n        mechanics_fail(\n            f"donor {clip}: the thinker returned "\n            f"{0 if hs is None else len(hs)} hidden state "\n            f"tensors, not the {len(LAYERS) + 1} the layer list "\n            f"implies")\n    states = [hs[k][0].detach().to("cpu", torch.float16)\n              for k in range(len(hs))]\n    del out, inputs\n    free_memory()\n    print(f"  captured {clip} under the "\n          f"{TARGET_FOR[CLIP_INFO[clip][\'group\']]} question: "\n          f"row {len(row)} positions, {len(pos)} audio, "\n          f"{len(states)} state tensors")\n    return {"clip": clip, "row": row, "positions": pos,\n            "states": states}\n\nCAPTURES = {}\nfor clip in [\'tone_00\', \'tone_01\', \'tone_02\', \'tone_03\', \'tone_04\', \'tone_05\', \'tone_06\', \'tone_07\']:\n    CAPTURES[(clip, PROMPT_BEEPS)] = capture(clip, PROMPT_BEEPS)\nfor clip in [\'silence_00\', \'silence_01\', \'silence_02\', \'silence_03\', \'silence_04\', \'silence_05\', \'silence_06\', \'silence_07\'][:4]:\n    CAPTURES[(clip, PROMPT_BEEPS)] = capture(clip, PROMPT_BEEPS)\nfor clip in [\'speech_00\', \'speech_01\', \'speech_02\', \'speech_03\']:\n    CAPTURES[(clip, PROMPT_SPEECH)] = capture(clip, PROMPT_SPEECH)\nprint(f"captures held: {len(CAPTURES)} donor runs")\n\n# --- The patch --------------------------------------------------------------------------------\n# One patched run: build the recipient\'s inputs, set a\n# forward hook on decoder layer L, and generate. The hook\n# fires on every forward through that layer; it patches\n# only the full length forward (the prefill that decides\n# the first generated token) and leaves the one token\n# decode steps alone, guarded by the sequence length. The\n# patch overwrites, at the chosen positions, the layer\'s\n# output with the donor\'s output at the same layer, donor\n# position by donor position in order. The hook is always\n# removed, answer or error.\ndef patched_generate(recipient, prompt, donor_cap, layer, mode,\n                     max_tokens, want_yes):\n    inputs = build_inputs(recipient, prompt)\n    row = [int(x) for x in inputs.input_ids[0].tolist()]\n    if len(row) != len(donor_cap["row"]):\n        mechanics_fail(\n            f"recipient {recipient} has a {len(row)} position "\n            f"row but donor {donor_cap[\'clip\']} has "\n            f"{len(donor_cap[\'row\'])}; a patch lines positions "\n            f"up one to one, so these two runs cannot be paired")\n    recip_audio = audio_positions(row)\n    if len(recip_audio) != EXPECTED_AUDIO_POSITIONS:\n        mechanics_fail(\n            f"recipient {recipient} has {len(recip_audio)} "\n            f"audio positions, not {EXPECTED_AUDIO_POSITIONS}")\n    if mode == "audio":\n        recip_pos = recip_audio\n        donor_pos = donor_cap["positions"]\n    elif mode == "nonaudio":\n        dset = set(donor_cap["positions"])\n        donor_pos = [i for i in range(len(donor_cap["row"]))\n                     if i not in dset]\n        rset = set(recip_audio)\n        recip_pos = [i for i in range(len(row)) if i not in rset]\n    else:\n        mechanics_fail(f"unknown patch mode {mode!r}")\n    if len(recip_pos) != len(donor_pos):\n        mechanics_fail(\n            f"recipient {recipient} offers {len(recip_pos)} "\n            f"{mode} positions but donor {donor_cap[\'clip\']} "\n            f"offers {len(donor_pos)}; the patch cannot line up")\n    donor_vec = donor_cap["states"][layer + 1][donor_pos]\n    full_len = len(row)\n    pos_index = torch.tensor(recip_pos, dtype=torch.long)\n    target_layer = LAYERS[layer]\n\n    def hook(module, args, output):\n        is_tuple = isinstance(output, tuple)\n        is_list = isinstance(output, list)\n        hs = output[0] if (is_tuple or is_list) else output\n        if hs.shape[1] != full_len:\n            return output   # a decode step, not the prefill\n        new = hs.clone()\n        new[0, pos_index.to(hs.device), :] = donor_vec.to(\n            device=hs.device, dtype=hs.dtype)\n        if is_tuple:\n            return (new,) + output[1:]\n        if is_list:\n            return [new] + list(output[1:])\n        return new\n\n    handle = target_layer.register_forward_hook(hook)\n    try:\n        decoded, yp = generate_from_inputs(inputs, max_tokens,\n                                           want_yes)\n    finally:\n        handle.remove()\n    del inputs\n    free_memory()\n    return decoded, yp\n\n# --- The sweep, checkpointed ---------------------------------------------------------------------\n# Rows are appended to patching_results.csv in both homes\n# the moment each patched run finishes, so an interruption\n# loses at most the run in flight. A fresh battery starts\n# clean: an existing results file is first renamed aside\n# with a timestamp, never silently overwritten and never\n# duplicated into.\nRESULTS_NAME = "patching_results.csv"\nfor p in out_paths(RESULTS_NAME):\n    if os.path.exists(p) and os.path.getsize(p) > 0:\n        aside = p + ".partial-" + datetime.datetime.now().strftime(\n            "%Y%m%d-%H%M%S")\n        try:\n            os.rename(p, aside)\n            print(f"note: an earlier {RESULTS_NAME} was moved "\n                  f"aside to {aside}; this battery starts clean")\n        except Exception as e:\n            print(f"could not move {p} aside ({str(e)[:120]}); "\n                  f"appends would mix runs, so the run stops")\n            sys.exit(12)\n\nappend_warned = set()\n\ndef append_results(new_rows):\n    for p in out_paths(RESULTS_NAME):\n        try:\n            exists = os.path.exists(p) and os.path.getsize(p) > 0\n            with open(p, "a", newline="") as fh:\n                w = csv.DictWriter(fh, fieldnames=RESULTS_FIELDS)\n                if not exists:\n                    w.writeheader()\n                w.writerows(new_rows)\n        except Exception as e:\n            if p not in append_warned:\n                append_warned.add(p)\n                print(f"WARNING: could not append results to "\n                      f"{p} ({str(e)[:120]}); the other home "\n                      f"still receives every row")\n\nresults_count = 0\ntone_lifts = {}   # layer -> list of (patched yes - recipient baseline)\n\ndef run_sweep(direction, pairs, layers, prompt, mode="audio"):\n    global results_count\n    print("=" * 70)\n    print(f"SWEEP: {direction}, {len(pairs)} pairs x "\n          f"{len(layers)} layers, mode {mode}")\n    for donor, recipient in pairs:\n        donor_cap = CAPTURES[(donor, prompt)]\n        for layer in layers:\n            decoded, yp = patched_generate(\n                recipient, prompt, donor_cap, layer, mode,\n                PRESENCE_MAX_TOKENS, True)\n            row_out = {\n                "pair": f"{donor}>{recipient}", "donor": donor,\n                "recipient": recipient, "direction": direction,\n                "layer": layer, "yes_score": f"{yp:.4f}",\n                "answer_word": decoded}\n            append_results([row_out])\n            results_count += 1\n            print(f"  {direction} {donor}>{recipient} layer "\n                  f"{layer}: yes {yp:.4f}, answer {decoded!r}")\n            if direction == "tone_to_silence":\n                lift = yp - baseline_yes[recipient]\n                tone_lifts.setdefault(layer, []).append(lift)\n    print(f"  {direction} done; results rows so far: "\n          f"{results_count}")\n\nALL_LAYERS = list(range(len(LAYERS)))\nrun_sweep("tone_to_silence",\n          [(t, s) for t, s in zip([\'tone_00\', \'tone_01\', \'tone_02\', \'tone_03\', \'tone_04\', \'tone_05\', \'tone_06\', \'tone_07\'],\n                                  [\'silence_00\', \'silence_01\', \'silence_02\', \'silence_03\', \'silence_04\', \'silence_05\', \'silence_06\', \'silence_07\'])],\n          ALL_LAYERS, PROMPT_BEEPS)\nrun_sweep("speech_to_silence",\n          [(sp, sil) for sp, sil in zip([\'speech_00\', \'speech_01\', \'speech_02\', \'speech_03\'],\n                                        [\'silence_00\', \'silence_01\', \'silence_02\', \'silence_03\', \'silence_04\', \'silence_05\', \'silence_06\', \'silence_07\'])],\n          ALL_LAYERS, PROMPT_SPEECH)\nrun_sweep("silence_to_tone",\n          [(sil, t) for sil, t in zip([\'silence_00\', \'silence_01\', \'silence_02\', \'silence_03\', \'silence_04\', \'silence_05\', \'silence_06\', \'silence_07\'],\n                                      [\'tone_00\', \'tone_01\', \'tone_02\', \'tone_03\', \'tone_04\', \'tone_05\', \'tone_06\', \'tone_07\'])][:4],\n          ALL_LAYERS, PROMPT_BEEPS)\nrun_sweep("tone_to_tone",\n          [(t, [\'tone_00\', \'tone_01\', \'tone_02\', \'tone_03\', \'tone_04\', \'tone_05\', \'tone_06\', \'tone_07\'][(i + 1) % len([\'tone_00\', \'tone_01\', \'tone_02\', \'tone_03\', \'tone_04\', \'tone_05\', \'tone_06\', \'tone_07\'])])\n           for i, t in enumerate([\'tone_00\', \'tone_01\', \'tone_02\', \'tone_03\', \'tone_04\', \'tone_05\', \'tone_06\', \'tone_07\'][:4])],\n          ALL_LAYERS, PROMPT_BEEPS)\nrun_sweep("wrong_positions",\n          [(t, s) for t, s in zip([\'tone_00\', \'tone_01\', \'tone_02\', \'tone_03\', \'tone_04\', \'tone_05\', \'tone_06\', \'tone_07\'],\n                                  [\'silence_00\', \'silence_01\', \'silence_02\', \'silence_03\', \'silence_04\', \'silence_05\', \'silence_06\', \'silence_07\'])][:4],\n          [0, 7, 14, 21, 28, 35], PROMPT_BEEPS, mode="nonaudio")\nprint(f"sweep complete: {results_count} patched runs recorded "\n      f"(the design\'s count is {744})")\nif results_count != 744:\n    print("WARNING: the recorded row count differs from the "\n          "design\'s count; the CSV is the record, and this "\n          "warning is part of it")\n\n# --- The best-layer open-form check ---------------------------------------------------------------\n# The layer where the tone patch lifted the silence\n# recipients\' yes scores most, on average over the 8 pairs,\n# against each recipient\'s own unpatched baseline. At that\n# one layer, the 8 patched silence runs answer the open\n# question: does the name arrive with the detection, or\n# not? Saved to patching_openform.csv.\nprint("=" * 70)\nprint("BEST LAYER: mean tone_to_silence lift per layer, over "\n      "the recipients\' baselines")\nmean_lifts = {}\nfor layer in ALL_LAYERS:\n    lifts = tone_lifts.get(layer, [])\n    mean_lifts[layer] = (sum(lifts) / len(lifts)) if lifts \\\n        else float("nan")\n    print(f"  layer {layer}: mean lift "\n          f"{mean_lifts[layer]:+.4f} over {len(lifts)} pairs")\nbest_layer = max(ALL_LAYERS, key=lambda l: mean_lifts[l])\nprint(f"best layer: {best_layer} (mean lift "\n      f"{mean_lifts[best_layer]:+.4f})")\nopen_rows = []\nfor tone in [\'tone_00\', \'tone_01\', \'tone_02\', \'tone_03\', \'tone_04\', \'tone_05\', \'tone_06\', \'tone_07\']:\n    if (tone, PROMPT_OPEN) not in CAPTURES:\n        CAPTURES[(tone, PROMPT_OPEN)] = capture(tone, PROMPT_OPEN)\nfor tone, silence in zip([\'tone_00\', \'tone_01\', \'tone_02\', \'tone_03\', \'tone_04\', \'tone_05\', \'tone_06\', \'tone_07\'], [\'silence_00\', \'silence_01\', \'silence_02\', \'silence_03\', \'silence_04\', \'silence_05\', \'silence_06\', \'silence_07\']):\n    decoded, _ = patched_generate(\n        silence, PROMPT_OPEN, CAPTURES[(tone, PROMPT_OPEN)],\n        best_layer, "audio", OPEN_MAX_TOKENS, False)\n    open_rows.append({"pair": f"{tone}>{silence}",\n                      "layer": best_layer, "answer": decoded})\n    print(f"  open patched {tone}>{silence} at layer "\n          f"{best_layer}: {decoded!r}")\n\ndef write_openform(p):\n    with open(p, "w", newline="") as fh:\n        w = csv.DictWriter(fh, fieldnames=OPENFORM_FIELDS)\n        w.writeheader()\n        w.writerows(open_rows)\nsave_both("patching_openform.csv", write_openform)\nunpatched_silence_open = next(\n    (r["answer"] for r in baseline_rows\n     if r["clip"] == "silence_00" and r["form"] == "open_p3"),\n    "")\nprint(f"for comparison, the unpatched silence_00 open answer "\n      f"from the baselines: {unpatched_silence_open!r}")\n\n# --- Closing prints -----------------------------------------------------------------------------------\nprint()\nprint("=" * 70)\nprint("patching done")\nprint(f"model: {MODEL_NAME} in {LOAD_MODE} form; layers patched: "\n      f"{LAYER_PATH}, {len(LAYERS)} layers; audio token id "\n      f"{AUDIO_TOKEN_ID}, {EXPECTED_AUDIO_POSITIONS} positions "\n      f"per row")\nprint("Pre-registered predictions from the tracing plan, quoted")\nprint("exactly. This notebook prints no verdict on them; the")\nprint("read happens outside:")\nprint("  Tone: \\"Patching rescues the yes/no answer at mid "\n      "layers but not the open form (naming is a later step "\n      "than detection).\\"")\nprint("  Speech: \\"Patching does NOT fix the denial, because "\n      "nothing is missing: the loss sits in how the presence "\n      "answer itself is formed, past the information.\\"")\nprint("Outputs: patching_load_log.txt, patching_token_check.txt,")\nprint("patching_baselines.csv, patching_results.csv,")\nprint("patching_openform.csv.")\n'
with open("/content/qwen_patching_run.py", "w") as fh:
    fh.write(RUNNER_SOURCE)
print("runner written:", "/content/qwen_patching_run.py", len(RUNNER_SOURCE), "chars")

In [ ]:
# --- Check mode: proof, candidates, layers, token check, feed proof -------------------
# Runs the runner under this session's own Python and streams
# its output here live. A nonzero exit code means the proof, a
# loading candidate, the layer resolution, the token check, or
# the feed proof failed, and the run stops in this cell. The
# first check run downloads the model files once; that takes a
# while and is normal.
import subprocess, sys
proc = subprocess.run([sys.executable, "/content/qwen_patching_run.py", "check"])
print()
print("check exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE CHECK RUN FAILED. The runner's own output above says")
    print("where: a proof line, a loading candidate, the layer resolution,")
    print("the token check, or the feed proof.")
    print("Do not run Step 4. Show this output to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: runner check failed")
print("check passed: proof lines right, model loaded, decoder layers")
print("resolved, token check passed with the audio positions derived")
print("and counted, feed proof passed on all 20 clips. Go on to Step 4.")

## Step 4 · Run the surgery battery

The next cell runs the runner in **battery mode**: proof,
candidates, layer resolution, the token check, and the
feed proof again, then five stages.

**Baselines.** Every clip runs unpatched under its own
question: tone and silence under the beeping question,
speech under the speaking question. The yes score (the
probability of a yes token at the first generated
position, Series Two's scorer) and the answer word land
in `patching_baselines.csv`, with the open question's
answers on tone_00, silence_00, and speech_00 as context
rows.

**Captures.** Sixteen donor runs: the 8 tone clips and the
first 4 silence clips under the beeping question, the 4
speech clips under the speaking question. Each capture
saves the donor's full hidden states from one frozen
forward, with its row and its audio positions. Donor and
recipient in a direction always share the direction's
question, so a patch moves sound information, never
question information.

**The sweep.** Patched runs, one prefill and a first-token
read each, the hook set on one decoder layer at a time:

- tone into silence, 8 pairs, layers 0 to 35 (288 runs);
- speech into silence, 4 pairs, all layers (144 runs);
- silence into tone, 4 pairs, all layers (144 runs): the
  reverse control, where the score should fall;
- tone into the next tone clip, 4 pairs, all layers (144
  runs): the sham control, where the sound is already
  present and little should change;
- tone into silence at the non-audio positions only,
  4 pairs, layers 0, 7, 14, 21, 28, 35 (24 runs): the
  wrong-positions control, where no rescue is expected.

744 patched runs in all. Every run's row (pair, donor,
recipient, direction, layer, yes score, answer word) is
appended to `patching_results.csv` the moment it
finishes, so an interruption loses at most the run in
flight. A fresh battery first moves any earlier results
file aside with a timestamp, so runs never mix.

**The best-layer open-form check.** The runner averages
the tone patch's lift per layer (patched score minus each
recipient's own baseline, over the 8 pairs) and names the
best layer. At that layer, the 8 patched silence runs
answer the open question, and their answers land in
`patching_openform.csv`, with the unpatched silence_00
open answer printed beside them for comparison.

**The closing prints.** The runner ends by printing
"patching done", the pre-registered predictions from the
tracing plan quoted exactly, and the output filenames. It
prints no verdict on the predictions; the read happens
outside this notebook.

In [ ]:
# --- Battery mode: the surgery battery ---------------------------------------------------
# Same runner, same session Python, streamed live. The runner
# saves its outputs itself, to the Drive tracing folder and
# to /content. A nonzero exit code means the token check
# failed, the feed proof hard-stopped the run, or a patching
# mechanic failed (the banners above say which); the results
# CSV holds every patched run that finished before the stop.
import subprocess, sys
proc = subprocess.run([sys.executable, "/content/qwen_patching_run.py", "battery"])
print()
print("battery exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE BATTERY RUN DID NOT FINISH CLEAN. Read the runner's")
    print("output above: the token check failed, the feed proof")
    print("hard-stopped the run, or a patching mechanic failed. The")
    print("results CSV holds every run that finished. Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: battery run not clean")
print("battery done: baselines taken, captures held, the sweep and")
print("its controls recorded, the best-layer open-form check saved.")

## Closing · What to send back

The files in the `tracing-qwen3b` Drive folder are the
record of this run:

- `patching_load_log.txt`: what loaded, and the full
  traceback of any candidate that failed.
- `patching_token_check.txt`: the histograms the audio
  positions were derived from, the settled id, and the
  counts. Read this before any result: it is the proof of
  which positions every capture and patch worked on.
- `patching_baselines.csv`: every clip's unpatched yes
  score and answer word, plus the three open answers.
- `patching_results.csv`: the 744 patched runs, appended
  as they finished.
- `patching_openform.csv`: the 8 patched silence runs'
  open answers at the best layer.

Send them back, and the surgery gets its read next to the
helpers and the attention watch, against the predictions
the plan registered before any of it was built. The
notebook itself prints no verdict.